# Refinement runs - baselines, utility, untargeted 

The graph is already built (`kg/hcm100`, 856/910) and the budget frontier is done.
This notebook adds the remaining experiments in one pass:
- **R1** reconnect (server + retry wrappers) - run after any disconnect
- **R2** the three [P7] baselines vs no-defense vs DP (targeted C3)
- **R3** answer utility (ROUGE) for each defense
- **R4** the untargeted attack
- **R5** backup logs to your computer

Every run is resumable (skips completed queries) and every LLM call retries 3x,
so a dropped connection costs a pause, not the session. If Colab resets entirely,
just re-run R1 then continue where you were.

## R1. Reconnect [run first, and after any disconnect]

Restarts Ollama with both models pinned, reloads the graph from disk, and installs
the retry wrappers. Nothing is rebuilt.

In [ ]:
%%bash
pkill -9 -f ollama 2>/dev/null; sleep 3
which ollama >/dev/null 2>&1 || (apt-get -qq install -y zstd pciutils lshw >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1)
nohup env OLLAMA_KEEP_ALIVE=-1 ollama serve > /tmp/ollama.log 2>&1 &
sleep 8
curl -s http://localhost:11434/api/generate -d '{"model":"qwen2.5:7b","prompt":"hi","options":{"num_ctx":4096},"keep_alive":-1,"stream":false}' >/dev/null
curl -s http://localhost:11434/api/embeddings -d '{"model":"nomic-embed-text","prompt":"hi","keep_alive":-1}' >/dev/null
echo "--- both models should show ---"; curl -s http://localhost:11434/api/ps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys; ROOT='/content/drive/MyDrive/graphrag_defense'; sys.path.insert(0, ROOT)
!pip install -q requests networkx numpy datasets pandas rouge-score >/dev/null 2>&1

from grd import config as C, data, attack, metrics, kg_build as kb, retriever as rt, run_b, utility
from grd.defenses import NoDefense, DPRetrieval, SystemPromptRule, SimilarityThreshold, Summarization
import grd.run_b as R, grd.utility as U
C.ensure_dirs(); C.DP_CANDIDATE_POOL = 30

# retry wrappers (the fix that stops disconnects from erroring queries)
import time, requests
def gen_retry(prompt, system='', model=None, host=None, num_ctx=4096, timeout=600):
    for _ in range(3):
        try:
            r=requests.post(f'{C.OLLAMA_HOST}/api/generate', json={'model':C.LLM_MODEL,'prompt':prompt,'system':system,'options':{'temperature':0.0,'num_ctx':num_ctx,'num_predict':512},'keep_alive':-1,'stream':False}, timeout=timeout)
            r.raise_for_status(); return r.json()['response']
        except Exception as e: last=e; time.sleep(5)
    raise last
def emb_retry(text, model=None, host=None, timeout=120):
    for _ in range(3):
        try:
            r=requests.post(f'{C.OLLAMA_HOST}/api/embeddings', json={'model':C.EMB_MODEL,'prompt':text,'keep_alive':-1}, timeout=timeout)
            r.raise_for_status(); return r.json()['embedding']
        except Exception: time.sleep(5)
    raise RuntimeError('embed failed')
kb.ollama_generate=gen_retry; kb.ollama_embed=emb_retry; rt.ollama_embed=emb_retry
R.retrieve=lambda q,store,top_n=None: rt.retrieve(q,store,top_n=top_n)
U.retrieve=lambda q,store,top_n=None: rt.retrieve(q,store,top_n=top_n)

# reload graph
store = kb.KGStore(f'{ROOT}/kg/hcm100')
N_ENT, N_REL, ALL_ENT, ALL_REL = run_b.graph_sets(store)
docs, records, idx = data.load_subset(n_docs=100)
qa = data.build_benign_qa(records)   # deterministic; reuses saved set if present
print('reconnected. graph', N_ENT, N_REL, '| benign QA', len(qa))

In [ ]:
%%bash
ollama pull qwen2.5:7b
ollama pull nomic-embed-text
echo "--- pinning ---"
curl -s http://localhost:11434/api/generate -d '{"model":"qwen2.5:7b","prompt":"hi","options":{"num_ctx":4096},"keep_alive":-1,"stream":false}' >/dev/null
curl -s http://localhost:11434/api/embeddings -d '{"model":"nomic-embed-text","prompt":"hi","keep_alive":-1}' >/dev/null
echo "--- api/ps ---"; curl -s http://localhost:11434/api/ps

In [ ]:
!curl -s http://localhost:11434/api/ps

## R2. The three [P7] baselines vs no-defense vs DP (targeted, C3)

NQ=40 to match the frontier. The Summarization baseline needs a summarizer, wired
below. This fills the head-to-head comparison table.

In [ ]:
import os, json
from grd import config as C
print("files in logs dir matching cmp_:")
for f in sorted(os.listdir(C.LOG_DIR)):
    if f.startswith('cmp_'):
        p = f'{C.LOG_DIR}/{f}'
        n = sum(1 for _ in open(p))
        # count errors
        errs = sum(1 for l in open(p) if json.loads(l).get('error'))
        print(f'  {f}: {n} lines, {errs} errors')

In [ ]:
import os
from grd import config as C
for tag in ['none','sysprompt','threshold','summarize','dp_b10']:
    p = f'{C.LOG_DIR}/cmp_targeted_C3_{tag}.jsonl'
    if os.path.exists(p): os.remove(p); print('deleted', tag)

In [ ]:
NQ = 40
summ = Summarization(summarize_fn=lambda t: kb.summarize_text(t))
conditions = {
    'none':       NoDefense(),
    'sysprompt':  SystemPromptRule(),
    'threshold':  SimilarityThreshold(tau=0.5),
    'summarize':  summ,
    'dp_b10':     DPRetrieval(eps_ent=1.0, eps_rel=1.0, k=10, user_budget=10),
}
runs=[]
for tag, dfn in conditions.items():
    name=f'cmp_targeted_C3_{tag}'
    run_b.run(store, attack.build_queries('targeted','C3')[:NQ], name, defense=dfn,
              use_cache=isinstance(dfn, DPRetrieval))
    runs.append(name)
metrics.summarize_runs(runs, N_ENT, N_REL, C.LOG_DIR)

## R3. Answer utility (ROUGE) for each defense

Runs the benign QA set through each defense as one session (so the budget is
consumed) and scores ROUGE-L against the reference answers. Report alongside R2:
this is the utility axis of the privacy-utility story.

In [ ]:
summ = Summarization(summarize_fn=lambda t: kb.summarize_text(t))
conditions = {
    'none':       NoDefense(),
    'sysprompt':  SystemPromptRule(),
    'threshold':  SimilarityThreshold(tau=0.5),
    'summarize':  summ,
    'dp_b10':     DPRetrieval(eps_ent=1.0, eps_rel=1.0, k=10, user_budget=10),
}
print('conditions ready:', list(conditions))

In [ ]:
for tag, dfn in conditions.items():
    U.run_utility(store, qa, tag, defense=dfn, use_cache=isinstance(dfn, DPRetrieval))
utility.score_utility(list(conditions.keys()))

In [ ]:
import json
from grd import config as C
p = f'{C.LOG_DIR}/util_dp_b10.jsonl'
errs = [json.loads(l) for l in open(p) if json.loads(l).get('error')]
print('errors:', len(errs))
if errs: print('first error:', errs[0]['error'][:200])

In [ ]:
import os
os.remove(f'{C.LOG_DIR}/util_dp_b10.jsonl')

# make sure server is alive first
!curl -s http://localhost:11434/api/ps

U.run_utility(store, qa, 'dp_b10', defense=DPRetrieval(eps_ent=1.0, eps_rel=1.0, k=10, user_budget=10), use_cache=True)
utility.score_utility(['none','sysprompt','threshold','summarize','dp_b10'])

## R4. Untargeted attack (no-defense vs budget sweep)

Shows the budget also bounds untargeted extraction, not just targeted.

In [ ]:
runs=['attack_untargeted_C3_none']
run_b.run(store, attack.build_queries('untargeted','C3')[:40], 'attack_untargeted_C3_none', defense=NoDefense())
for B in [5,10,20,40]:
    name=f'attack_untargeted_C3_b{B}'
    run_b.run(store, attack.build_queries('untargeted','C3')[:40], name,
              defense=DPRetrieval(eps_ent=1.0,eps_rel=1.0,k=10,user_budget=B), use_cache=True)
    runs.append(name)
metrics.summarize_runs(runs, N_ENT, N_REL, C.LOG_DIR)

## R5. Back up all logs + graph to your computer

Do this whenever you have new results, so a session wipe never costs them.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/logs_backup','zip', C.LOG_DIR)
shutil.make_archive('/content/kg_backup','zip', f'{ROOT}/kg/hcm100')
files.download('/content/logs_backup.zip')
files.download('/content/kg_backup.zip')

Paste the R2, R3, and R4 tables back and I will drop them into the paper:
R2+R3 become the head-to-head comparison table (privacy and utility together),
R4 becomes the untargeted frontier. That completes the results section.